# **Laboratorio Clase 42: Tracking en video**

Profesor: Carlos Aspillaga

Al igual que en laboratorios anteriores, debe responder el laboratorio de forma individual y asegurarse de entregar con todas las celdas ejecutadas.
Habrá un bonus de 1 décima por orden y redacción a criterio del ayudante corrector.

In [ ]:
# ==========================================
# Phase 1: Environment and Dependency Setup
# ==========================================
!pip install -U ultralytics supervision roboflow opencv-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.3/373.3 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 5.0.0.93
    Uninstalling 

In [ ]:
import torch
import cv2
import numpy as np
import os
import urllib.request
from collections import defaultdict
from IPython.display import HTML, display
from base64 import b64encode

# Verify PyTorch CUDA Allocation for hardware-accelerated inference
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Allocated Accelerator: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Allocated Accelerator: Tesla T4


In [ ]:
!wget https://github.com/intel-iot-devkit/sample-videos/raw/master/people-detection.mp4

--2026-08-05 17:58:32--  https://github.com/intel-iot-devkit/sample-videos/raw/master/people-detection.mp4
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/people-detection.mp4 [following]
--2026-08-05 17:58:33--  https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/people-detection.mp4
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5482579 (5.2M) [application/octet-stream]
Saving to: ‘people-detection.mp4’

people-detection.mp 100%[===================>]   5.23M  --.-KB/s    in 0.02s   

2026-08-05 17:58:34 (327 MB/s) - ‘people-detecti

In [ ]:
# ==========================================
# Phase 2: YOLO26 Tracking Execution
# ==========================================
from ultralytics import YOLO

# Instantiate the SOTA YOLO26 nano model.
# Its NMS-free architecture ensures deterministic, low-latency tracking suitable for edge deployment.
model = YOLO("yolo26n.pt")

# Initialize video capture using OpenCV
cap = cv2.VideoCapture("people-detection.mp4")

# Safety check!
if not cap.isOpened():
    raise ValueError(f"CRITICAL ERROR: OpenCV could not open 'people-detection.mp4'. The file is missing or corrupted.")

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Second safety check!
if width == 0 or height == 0:
    raise ValueError("CRITICAL ERROR: Video dimensions are 0x0. The video stream is invalid.")

output_path_yolo = "yolo26_tracking_output.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_yolo = cv2.VideoWriter(output_path_yolo, fourcc, fps, (width, height))

# Initialize a default dictionary to persistently store mathematical trajectory history
track_history = defaultdict(list)

print("Executing YOLO26 spatio-temporal tracking pipeline...")
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # Extract the first (and only) Results object from the returned list
    results = model.track(frame, persist=True, tracker="botsort.yaml", verbose=False)[0]

    # 'results' is a Results object
    if results.boxes.id is not None:
        boxes = results.boxes.xywh.cpu() # x_center, y_center, width, height
        track_ids = results.boxes.id.int().cpu().tolist()

        # Superimpose the fundamental bounding boxes onto the frame tensor
        annotated_frame = results.plot()

        # Iterate through detected IDs to draw historical trajectory lines
        for box, track_id in zip(boxes, track_ids):
            x, y, w, h = box
            track = track_history[track_id]
            track.append((float(x), float(y))) # Append current spatial center point

            # Constrain the trajectory visualizer to the last 30 frames to prevent tensor memory bloat
            if len(track) > 30:
                track.pop(0)

            # Utilize OpenCV to render the trajectory polyline
            points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
            cv2.polylines(annotated_frame, [points], isClosed=False, color=(0, 255, 255), thickness=2)

        out_yolo.write(annotated_frame)
    else:
        out_yolo.write(frame)

cap.release()
out_yolo.release()
print(f"YOLO26 Tracking finalized. Video synthesized to {output_path_yolo}")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Executing YOLO26 spatio-temporal tracking pipeline...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 193ms
Prepared 1 package in 42ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

YOLO26 Tracking finalized. Video synthesized to yolo26_tracking_output.mp4


In [ ]:
!ls

people-detection.mp4  sample_data  yolo26n.pt  yolo26_tracking_output.mp4


In [ ]:
!ffmpeg -i yolo26_tracking_output.mp4 -vcodec libx264 compressed_yolo26_tracking_output.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
# Read the binary stream and encode to base64 for direct browser injection
mp4_data = open("compressed_yolo26_tracking_output.mp4",'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4_data).decode()

# Render utilizing IPython display utilities
display(HTML(f"""<video width=800 controls><source src="{data_url}" type="video/mp4"></video>"""))

In [ ]:
# ==========================================
# Phase 3: Composite Open-Vocabulary Segmentation
# (YOLO-World + SAM 2 Pipeline)
# ==========================================
from ultralytics import YOLOWorld, SAM
import cv2
import numpy as np

print("Loading Open-Vocabulary Detector and Segmentation models...")

# 1. Initialize YOLO-World (Vision-Language Zero-Shot Detector)
# This handles the "Text Prompt" portion of the pipeline.
yolo_world = YOLOWorld("yolov8s-worldv2.pt")

# Define our open-vocabulary concepts. YOLO-World will dynamically compile these into its detection head.
# You can change these to anything you want to detect in the video!
custom_prompts = ["person"]
yolo_world.set_classes(custom_prompts)

# 2. Initialize SAM 2 (Segment Anything Model 2)
# We use the nano version for fast inference in Colab.
# This handles the pixel-level masking based on the bounding boxes provided by YOLO-World.
sam_model = SAM("sam2.1_t.pt")

# 3. Setup Video Stream
video_path = "people-detection.mp4"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError(f"CRITICAL ERROR: OpenCV could not open {video_path}.")

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

output_path_composite = "composite_segmentation_output.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_composite = cv2.VideoWriter(output_path_composite, fourcc, fps, (width, height))

print(f"Executing YOLO-World + SAM 2 Pipeline on concepts: {custom_prompts}...")

frame_count = 0
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    frame_count += 1
    if frame_count % 30 == 0:
        print(f"Processing frame {frame_count}...")

    # Step A: Concept Grounding (YOLO-World)
    # We use persist=True to maintain tracking IDs across frames
    yolo_results = yolo_world.track(frame, persist=True, verbose=False)[0]

    # Step B: Extract Spatial Coordinates
    # SAM requires bounding boxes in xyxy format (xmin, ymin, xmax, ymax)
    if yolo_results.boxes.id is not None and len(yolo_results.boxes.xyxy) > 0:
        boxes_xyxy = yolo_results.boxes.xyxy.cpu().numpy()

        # Step C: Prompted Segmentation (SAM 2)
        # We pass the bounding boxes directly to SAM as visual prompts
        sam_results = sam_model(frame, bboxes=boxes_xyxy, verbose=False)[0]

        # Superimpose the generated alpha-blended masks onto the frame
        # We also draw the bounding boxes and labels from YOLO-World for clarity
        annotated_frame = sam_results.plot(boxes=False) # Plot SAM masks
        annotated_frame = yolo_results.plot(img=annotated_frame) # Overlay YOLO boxes/labels

        out_composite.write(annotated_frame)
    else:
        # If YOLO-World didn't find our text prompts in this frame, write the raw frame
        out_composite.write(frame)

cap.release()
out_composite.release()
print(f"Pipeline finalized. Video synthesized to {output_path_composite}")



Loading Open-Vocabulary Detector and Segmentation models...
requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 37 packages in 749ms
Prepared 2 packages in 2.56s
Installed 2 packages in 1ms
 + clip==1.0 (from git+https://github.com/ultralytics/CLIP.git@488e81a6711eea7346872b46ea928b367da8889d)
 + ftfy==6.3.1

requirements: AutoUpdate success ✅ 3.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 108MiB/s]


Executing YOLO-World + SAM 2 Pipeline on concepts: ['person']...
Processing frame 30...
Processing frame 60...
Processing frame 90...
Processing frame 120...
Processing frame 150...
Processing frame 180...
Processing frame 210...
Processing frame 240...
Processing frame 270...
Processing frame 300...
Processing frame 330...
Processing frame 360...
Processing frame 390...
Processing frame 420...
Processing frame 450...
Processing frame 480...
Processing frame 510...
Processing frame 540...
Processing frame 570...
Pipeline finalized. Video synthesized to composite_segmentation_output.mp4


In [ ]:
!ffmpeg -y -i composite_segmentation_output.mp4 -vcodec libx264 compressed_composite_segmentation_output.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
# Read the binary stream and encode to base64 for direct browser injection
mp4_data = open("compressed_composite_segmentation_output.mp4",'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4_data).decode()

# Render utilizing IPython display utilities
display(HTML(f"""<video width=800 controls><source src="{data_url}" type="video/mp4"></video>"""))

**Actividad**

1. a) Corra Yolo con al menos 3 nuevos videos
 b) detecte los tipos de error más usuales. Describa si los errores hacen referencia a la detección o si hacen referencia a la asignación de identificadores.
 c) relacione con los conceptos de la clase

2. a) Corra el pipeline de SAM para al menos 3 videos (pueden ser mismos videos de la actividad 1). En alguno solicite tracking de un objeto únicos, en otro solicite el tracking de un objeto que sea múltiple (ej: 2 personas que aparecen simultáneamente en el video), y en otro solicite al modelo que haga tracking de más de un prompt simultáneamente.
b) Analice los resultados y comente los tipos de error que identificó. Describa si los errores hacen referencia a la detección o si hacen referencia a la asignación de identificadores.
 c) Relacione con los conceptos de la clase